# 04 — Perfis dos Clusters e Ações

Objetivos:
- Anexar `cluster_id` à visão cliente-atual
- Construir perfis (tamanho, percentuais, médias/medianas, distribuições)
- Nomear clusters com rótulos de negócio
- Criar ranking (valor/risco) com métrica transparente
- Responder explicitamente: recorrente e nunca acessou
- Exportar: `dataset_clusterizado.xlsx` e `resumo_clusters.xlsx`

In [1]:
from pathlib import Path
import sys

sys.path.append(str(Path('..').resolve()))


import numpy as np
import pandas as pd

from src.features import build_modeling_dataframe
from src.modeling import (
    cluster_profiles,
    compute_cluster_score,
    load_pipeline,
)

PROJECT_ROOT = Path('..').resolve()
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
REPORTS_DIR = PROJECT_ROOT / 'reports'
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

C:\Users\flavi\Anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [2]:
df_cliente = pd.read_parquet(PROCESSED_DIR / 'base_cliente_atual.parquet')
X, spec = build_modeling_dataframe(df_cliente)
pipe = load_pipeline(str(REPORTS_DIR / 'kmeans_pipeline.joblib'))
df_cliente.shape, X.shape

((45027, 36), (45027, 16))

## 1) Predição de cluster

In [3]:
df_cliente = df_cliente.copy()
df_cliente['cluster_id'] = pipe.predict(X)
df_cliente['cluster_id'].value_counts().sort_index()

cluster_id
0     9709
1    16126
2     5501
3     5475
4     4175
5     4041
Name: count, dtype: int64

## 2) Perfis e nomes de negócio

In [4]:
NOME_BY_CLUSTER = {
    4: 'Champions',
    0: 'Potenciais',
    5: 'Novos',
    1: 'Avulsos Engajados',
    3: 'Churn Iminente',
    2: 'Zumbis',
}

df_cliente['cluster_id'] = df_cliente['cluster_id'].astype(int)
names = (
    pd.DataFrame({'cluster_id': list(NOME_BY_CLUSTER.keys()), 'nome_cluster': list(NOME_BY_CLUSTER.values())})
    .sort_values('cluster_id')
    .reset_index(drop=True)
)

df_cliente['nome_cluster'] = df_cliente['cluster_id'].map(NOME_BY_CLUSTER)
if df_cliente['nome_cluster'].isna().any():
    missing = sorted(df_cliente.loc[df_cliente['nome_cluster'].isna(), 'cluster_id'].unique().tolist())
    raise ValueError(f'Clusters sem nome mapeado: {missing}')

names

,cluster_id,nome_cluster
0,0,Potenciais
1,1,Avulsos Engajados
2,2,Zumbis
3,3,Churn Iminente
4,4,Champions
5,5,Novos


In [5]:
profiles = cluster_profiles(
    df_cliente,
    cluster_col='cluster_id',
    numeric_cols=[c for c in ['log_acessos','dias_sem_acessar','recorrente','total_parcelas','n_transacoes_cliente','recencia_compra_dias','freq_compra_mensal','parcelado','nunca_acessou'] if c in df_cliente.columns],
    categorical_cols=[c for c in ['ativo','renovacao','pagamento_tratado','tipo_pagamento','faixa_inatividade','status'] if c in df_cliente.columns],
)
profiles['summary'].head(10)

,cluster_id,tamanho,pct_total,log_acessos_mean,log_acessos_median,dias_sem_acessar_mean,dias_sem_acessar_median,recorrente_mean,recorrente_median,total_parcelas_mean,...,tipo_pagamento__3,faixa_inatividade__0_7,faixa_inatividade__31_90,faixa_inatividade__91_180,faixa_inatividade__181_plus,status__APPROVED,status__CHARGEBACK,status__COMPLETE,status__PROTESTED,status__REFUNDED
1,1,16126,0.358141,2.303259,2.302585,112.480032,90.0,0.000000,0.0,1.000000,...,0.000620,0.057919,0.448034,0.356071,0.137976,0.046013,0.001302,0.947476,0.000062,0.005147
0,0,9709,0.215626,2.266657,2.197225,105.981872,88.0,1.000000,1.0,11.956844,...,0.000103,0.075703,0.445154,0.367803,0.111340,0.037182,0.000103,0.953754,0.000103,0.008858
2,2,5501,0.122171,0.000000,0.000000,0.000000,0.0,0.521542,1.0,6.714961,...,0.042174,1.000000,0.000000,0.000000,0.000000,0.601163,0.001454,0.330849,0.002909,0.063625
3,3,5475,0.121594,1.858737,1.791759,204.195616,201.0,0.842009,1.0,10.217717,...,0.000365,0.041461,0.125479,0.273242,0.559817,0.041096,0.001826,0.592511,0.001279,0.363288
4,4,4175,0.092722,3.044069,3.135494,165.522156,130.0,0.375808,0.0,5.103713,...,0.000479,0.020120,0.263473,0.370060,0.346347,0.000000,0.003353,0.984192,0.000000,0.012455
5,5,4041,0.089746,2.135862,2.079442,115.923039,94.0,0.353873,0.0,4.751794,...,0.904727,0.106904,0.351151,0.369958,0.171987,0.066568,0.000000,0.882950,0.000247,0.050235


## 3) Ranking (valor/risco)

Métrica transparente: score ponderado (engajamento + recorrência + renovação − inatividade − nunca_acessou).

In [6]:
rank = compute_cluster_score(df_cliente, cluster_col='cluster_id')
rank = rank.merge(names, on='cluster_id', how='left')
rank

,cluster_id,score_medio,ranking,nome_cluster
0,4,0.902426,1,Champions
1,0,0.366368,2,Potenciais
2,5,-0.010824,3,Novos
3,1,-0.138318,4,Avulsos Engajados
4,3,-0.165266,5,Churn Iminente
5,2,-0.753610,6,Zumbis


## 4) Pergunta 4: recorrente e nunca acessou

Filtro na visão cliente-atual: `recorrente==1` e `nunca_acessou==1`.

In [7]:
mask = (df_cliente.get('recorrente', 0).fillna(0) == 1) & (df_cliente.get('nunca_acessou', 0).fillna(0) == 1)
q4 = df_cliente.loc[mask].copy()
q4_size = len(q4)
q4_pct = q4_size / len(df_cliente)
q4_size, q4_pct

(3979, 0.08836920070180114)

## 5) Ações recomendadas por cluster (playbooks)

As recomendações abaixo são geradas a partir do perfil (engajamento/inatividade/recorrência/parcelamento). Ajuste fino pode ser feito após leitura do dicionário do case (PDF).

In [8]:
def recomendar_acao(row):
    nome = str(row['nome_cluster'])
    if nome == 'Champions':
        return 'Upsell/cross-sell, programa de indicação, early access; manter NPS alto com comunicações personalizadas.'
    if nome == 'Potenciais':
        return 'Elevar valor percebido e reduzir inatividade: campanhas por interesse/curso e check-ins por gatilho (14–30 dias sem acessar).'
    if nome == 'Novos':
        return 'Boas-vindas, checklist de primeiros passos, gatilhos de valor (primeira aula/primeiro resultado), acompanhamento D+3/D+7.'
    if nome == 'Avulsos Engajados':
        return 'Converter para recorrência: oferta de plano recorrente com proposta clara, bundles e condições especiais para upgrade.'
    if nome == 'Churn Iminente':
        return 'Playbook de retenção: diagnóstico de bloqueios, campanhas win-back, ofertas de pausa/alternativa de plano, CS 1:1 para top contas.'
    if nome == 'Zumbis':
        return 'Onboarding agressivo + nudges de ativação (sequência 7 dias), webinars, contato CS proativo; foco em ativação para reduzir cancelamento futuro.'
    return 'Ações de engajamento: trilhas de conteúdo, lembretes de progresso, campanhas por faixa de inatividade e tipo de pagamento.'

acoes = names.copy()
acoes['acoes_recomendadas'] = acoes.apply(recomendar_acao, axis=1)
acoes

,cluster_id,nome_cluster,acoes_recomendadas
0,0,Potenciais,Elevar valor percebido e reduzir inatividade: ...
1,1,Avulsos Engajados,Converter para recorrência: oferta de plano re...
2,2,Zumbis,Onboarding agressivo + nudges de ativação (seq...
3,3,Churn Iminente,Playbook de retenção: diagnóstico de bloqueios...
4,4,Champions,"Upsell/cross-sell, programa de indicação, earl..."
5,5,Novos,"Boas-vindas, checklist de primeiros passos, ga..."


## 6) Exportação obrigatória (Excel)
- `reports/dataset_clusterizado.xlsx`: base cliente-atual + cluster + nome_cluster + principais features
- `reports/resumo_clusters.xlsx`: tabelas agregadas + ranking + ações

In [9]:
main_cols = [c for c in ['cliente','cluster_id','nome_cluster','log_acessos','dias_sem_acessar','recorrente','total_parcelas','parcelado','nunca_acessou','renovacao','ativo','n_transacoes_cliente','recencia_compra_dias','freq_compra_mensal','tipo_pagamento','pagamento_tratado','status'] if c in df_cliente.columns]
dataset_clusterizado = df_cliente[main_cols].copy()

dataset_path = REPORTS_DIR / 'dataset_clusterizado.xlsx'
with pd.ExcelWriter(dataset_path, engine='xlsxwriter') as writer:
    dataset_clusterizado.to_excel(writer, sheet_name='cliente_atual_cluster', index=False)
    q4[main_cols].to_excel(writer, sheet_name='q4_recorrente_sem_acesso', index=False)

resumo_path = REPORTS_DIR / 'resumo_clusters.xlsx'
with pd.ExcelWriter(resumo_path, engine='xlsxwriter') as writer:
    profiles['summary'].to_excel(writer, sheet_name='perfil_resumo', index=False)
    profiles['sizes'].to_excel(writer, sheet_name='tamanhos', index=False)
    profiles['numeric'].to_excel(writer, sheet_name='numericas', index=False)
    profiles['categorical'].to_excel(writer, sheet_name='categoricas', index=False)
    rank.to_excel(writer, sheet_name='ranking', index=False)
    acoes.to_excel(writer, sheet_name='acoes', index=False)

df_cliente.to_parquet(PROCESSED_DIR / 'base_cliente_clusterizada.parquet', index=False)
dataset_path, resumo_path

(WindowsPath('C:/Users/flavi/Documents/GitHub/Customer_Sucess_Clustering/reports/dataset_clusterizado.xlsx'),
 WindowsPath('C:/Users/flavi/Documents/GitHub/Customer_Sucess_Clustering/reports/resumo_clusters.xlsx'))